# Conduction block with an anode–cathode configuration of the two sites

**Project:** FEM simulation of intraneural stimulation of the median nerve, block study
(walkthrough `docs/block.md`, decision D-12 in `docs/decisions.md`).
**Model:** patient UA01, the solved 3D model shown in the viewer. Only station −4 (contacts 1–6) is
solved; the distal contacts 43–48 are its mirror image (D-12).

**In one sentence.** The **proximal site** stimulates one fascicle with a short **cathodic** pulse
(this gives the sensation). The **distal site** drives **the same contacts** (their mirror partners)
as an **anode**, to stop the action potentials (APs) that the same pulse sends towards the hand,
where they would reach the reinnervated muscle. The notebook looks for conditions (current, pulse
shape) where the anode blocks the descending AP **without firing the fibre itself**.

**The supervisor's rule (2026-11-26):** the active contacts of the anode site are the same as those
activated on the cathode site. So the anode is not a free choice: with AS 2 as the cathode, the
anode is AS 44 (the same contact on the mirror station). What we can still choose is the anodal
**current** and the **shape and timing** of its pulse.

**How to run**
1. Prerequisites (once per computer, `docs/block.md` §4): NEURON installed,
   `results/mrg/model/nrnmech.dll` compiled, Python packages numpy, scipy, h5py, pandas, matplotlib.
2. The FEM profiles must exist: `results/block/UA01_fibre_profiles.h5` and
   `results/block/UA01_block_params.json`. They are written by
   `matlab -batch "setup_paths; export_block_profiles"`. Run it again after any change to
   `config/block_params.m`.
3. Open this notebook (Jupyter or VS Code) with the Python kernel that has NEURON and choose
   **Run All**. With 7 worker processes it takes about 3–6 min, most of it in section 8.
4. Outputs go to `results/block/anode_cathode/`: figures `*.png`, tables `*.csv` and `summary.json`.

If the parallel runs fail (an error raised inside a worker process), set `N_WORKERS = 1` in the
parameter cell (section 3) and run again. Everything then runs in this process, about 7 times slower.
If you edit `src/block/fibre_jobs.py` or `mrg_fibre.py`, restart the kernel: modules are imported
only once.

**Every figure has a "How to read this figure" note** just before the cell that draws it.

**Sections:** 1 background · 2 setup · 3 parameters · 4 data · 5 proximal stimulation (thresholds,
stimulation current, example fibre) · 6 why the anode fires the fibre · 7 map: ramp × current ·
8 population check of the best settings · 9 the textbook bipolar pair · 10 summary and limitations.

## 1. Background: what the anode is supposed to do, and why it is hard

**The configuration.** Two sites, 20.6 mm apart along the nerve:

| | proximal site, z = −10.3 mm (shoulder side) | distal site, z = +10.3 mm (hand side) |
|---|---|---|
| role | stimulation: gives the sensation | block: stops the APs going to the muscle |
| polarity | **cathode** (current drawn out of the tissue), short 50 µs pulse | **anode** (current pushed into the tissue), long pulse |
| contacts | a pattern aimed at one fascicle (here AS 2, inside fascicle F7) | **the same contacts**, mirrored (here AS 44), supervisor 2026-11-26 |

**How an electrode acts on a fibre: the activating function.** A myelinated fibre "feels" the
extracellular potential $V_e$ only at its nodes of Ranvier (the myelin insulates the rest). The
current the stimulus pushes into node $n$ is proportional to the second difference of $V_e$ along
the fibre:

$$ f_n = \frac{V_{e,n-1} - 2V_{e,n} + V_{e,n+1}}{\Delta x^2} $$

where $\Delta x$ is the internode length (1.15 mm for a 10 µm fibre). $f_n > 0$ **depolarises** node
$n$ and pushes it towards firing; $f_n < 0$ **hyperpolarises** it. In control terms, $V_e$ is an input
applied along a chain of nodes, and the discrete Laplacian (the second difference) is the input matrix
through which it reaches each node's membrane.

- Under a **cathode**, $V_e$ has a negative dip, so its second difference is positive at the centre.
  That is depolarisation, and the AP starts there. On both sides the curvature reverses, which gives
  two hyperpolarised lobes ("virtual anodes").
- Under an **anode** it is the opposite: the centre is hyperpolarised (the block we want), and there
  are **two depolarised side lobes, the "virtual cathodes"**, one or two internodes away.

**Anodal block.** If the centre is hyperpolarised enough while the AP arrives, the arriving AP cannot
bring those nodes to threshold, and it stops there. The anode must be ON while the AP passes. The
sites are 20.6 mm apart and the AP travels at ≈ 50 m/s, so it arrives ≈ 0.4 ms after the proximal
pulse. The anodal pulse therefore has to be long (≈ 1 ms), not 50 µs.

**Two ways the anode can fire the fibre itself:**
1. *Virtual-cathode excitation.* The side lobes depolarise. If the anodal current rises
   **abruptly**, they can fire an AP at the pulse onset, and that AP travels to the muscle (and to the
   brain). This is what happened in the first round (`docs/block.md` §6): the anode on AS 44 fired
   every excited fibre from ≈ 2 × the stimulation current, long before it blocked anything.
2. *Anode break.* During a long hyperpolarisation the inactivation gate of the Na channels recovers,
   so more channels are available than at rest. If the anode is switched off **abruptly**, the
   membrane rebounds and can fire.

**What this notebook tries.** The contacts are fixed by the supervisor's rule, so the free variables
are the current and the pulse shape. The remedy for both problems is to **ramp the anodal current**
(a trapezoidal pulse; quasi-trapezoidal pulses, Fang & Mortimer 1991, IEEE Trans Biomed Eng 38:168;
not yet in `Papers/`, to add to `docs/literature/` if we keep this result):
- a slow rise lets the membrane at the virtual cathodes **accommodate**: while the depolarisation
  builds up slowly, the Na channels inactivate and the K channels open, so the threshold rises with
  it and no AP starts;
- a slow fall avoids anode break.

As a reference we also run the **textbook bipolar pair** on the same two contacts: proximal cathode
and distal anode in one circuit, carrying the same current. With a few mm between the poles, as in
a cuff, this configuration gives unidirectional stimulation.

**Questions**
- **Q1** (section 6): why does the anode fire the fibre before it blocks it?
- **Q2** (section 7): does ramping the anode open a window of currents where it blocks without
  exciting?
- **Q3** (section 8): for the best settings, how many target fibres are blocked, how many fibres of
  other fascicles does the anode fire, and at what charge?
- **Q4** (section 9): does the textbook bipolar pair work with the two sites 20.6 mm apart?

Everything is simulated with the MRG fibre model (McIntyre, Richardson & Grill 2002) in NEURON, on
straight fibres inside the fascicles. The FEM potential of each contact is applied by superposition
(`docs/block.md` §3).

## 2. Setup

Imports and paths. The fibre model lives in `src/block/mrg_fibre.py`, which builds the MRG fibre in
NEURON and applies any combination of contact currents to it. The batch runner lives in
`src/block/fibre_jobs.py`: one fibre simulation is one "job", and the runner executes lists of jobs
on several processes. Read the docstrings of those two files for the details. This notebook only
decides **which** simulations to run and how to read them.

In [ ]:
# --- Standard library and the scientific stack -----------------------------------------------------
import json            # reads the parameter file written by MATLAB
import math
import os
import sys
import time            # wall-clock timing of the long cells
from pathlib import Path

import h5py            # reads the FEM potential profiles (HDF5 file written by MATLAB)
import numpy as np
import pandas as pd    # result tables
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import scipy.io        # reads data/electrode/sites.mat (the electrode layout)

# --- Where is the project? --------------------------------------------------------------------------
# Jupyter starts the kernel in the folder of the notebook (notebooks/): the project root is one level up.
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "block").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "block" / "mrg_fibre.py").exists(), \
    f"Cannot find the project root from {Path.cwd()}: open the notebook from the notebooks/ folder."

# --- NEURON -----------------------------------------------------------------------------------------
try:
    from neuron import h
except ImportError as err:
    raise ImportError("NEURON is not importable from this kernel. On Windows install it with the official "
                      "installer (docs/block.md §4), then start Jupyter from a NEW terminal so that the "
                      "PYTHONPATH set by the installer is seen.") from err
print("NEURON:", h.nrnversion())

# --- The project's own modules (src/block/) ---------------------------------------------------------
sys.path.insert(0, str(ROOT / "src" / "block"))
from mrg_fibre import MRG_TABLE        # MRG geometry table: internode length etc. per fibre diameter
import fibre_jobs                      # batch runner: one fibre simulation = one "job"

MECH_DIR = ROOT / "results" / "mrg" / "model"     # compiled AXNODE mechanism (nrnmech.dll)
BLOCK_DIR = ROOT / "results" / "block"             # inputs written by scripts/export_block_profiles.m
OUT_DIR = BLOCK_DIR / "anode_cathode"              # outputs of this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)
print("outputs     :", OUT_DIR)

## 3. Parameters

Two kinds of numbers:
- **Shared parameters** of the block study (fibre model, proximal pulse width, recruitment criterion,
  stimulation factor) are set in `config/block_params.m`. They are read here from the JSON copy that
  `export_block_profiles.m` writes next to the profiles, so this notebook and the script
  `run_block_study.py` use the same values.
- **The knobs of this exploration** (ramp times, current ranges, timing of the anode) are in the cell
  below. If one of these choices becomes a project decision, move it to `config/block_params.m` and
  log it in `docs/decisions.md`.

**Sign convention.** In a pattern `{contact: weight}`, the weight is the fraction of the channel
current that flows through that contact, and its sign is the polarity: negative = cathodic,
positive = anodic. Pulse amplitudes are always positive. So `{2: -1.0}` driven at 10 µA means
"10 µA cathodic on AS 2", and `{44: 1.0}` driven at 100 µA means "100 µA anodic on AS 44". The return
path is the grounded far boundary of the FEM model (the two end planes of the domain), except in the
bipolar pair of section 9, where the cathodic and anodic weights cancel.

**The anode pattern is derived, not chosen.** It is the proximal pattern with every contact replaced
by its mirror partner (section −k → +k) and the sign reversed (supervisor, 2026-11-26). To study
another fascicle, change only `PROX_PATTERN` and `TARGET_FASCICLE`: for example `{1: -1.0}` with
fascicle 2, or `{4: -1.0}` with fascicle 1. Only contacts 1–6 can be used, since they form the only
solved station.

In [ ]:
PATIENT = "UA01"

# Shared parameters = config/block_params.m, read from the JSON copy written by export_block_profiles.m
P = json.loads((BLOCK_DIR / f"{PATIENT}_block_params.json").read_text())
MRG = P["mrg"]                       # MRG fibre settings: diameter, length, time step, AP detection...

# -----------------------------------------------------------------------------------------------------
# Knobs of THIS notebook. Units: SI in the code (A, s, m); µA, ms, mm only in labels and tables.
# -----------------------------------------------------------------------------------------------------

# Proximal (stimulation) pattern {contact id: weight}, cathodic < 0. AS 2 lies inside fascicle F7
# (results/main3d, out.results.location); it is the pattern of the first round (docs/block.md §6).
# The distal ANODE pattern is built from it in section 4 (same contacts, mirrored, sign reversed).
PROX_PATTERN = {2: -1.0}
TARGET_FASCICLE = 7                  # the fascicle the proximal pattern is meant to excite

FIBRE_DIAM_um = MRG["fibreDiam_um"]  # 10 um by default (config); allowed values = keys of MRG_TABLE

# Timing. Every anode time is counted from the ONSET of the proximal pulse.
T_PROX_ON_s  = 1.5e-3                # proximal pulse onset. It must exceed the longest ramp, so that a
                                     # ramp can start BEFORE the proximal pulse, with the fibre at rest
PROX_WIDTH_s = P["pulse"]["stimWidth_s"]   # 50 us (Zelechowski 2020)
T_AFTER_s    = P["pulse"]["tAfter_s"]      # simulated time after the last pulse (APs reach the fibre ends)

# Anode window. The anode is at full amplitude (the "plateau") from PLATEAU_START to PLATEAU_END after
# the proximal onset. The plateau must contain the arrival of the AP at the distal site (~0.4 ms for
# 10 um fibres, measured in section 5). A rising ramp of length `rise` ENDS at PLATEAU_START; the
# falling ramp STARTS at PLATEAU_END.
PLATEAU_START_s  = 0.1e-3
PLATEAU_END_s    = 0.9e-3
RISE_TIMES_s     = [0.0, 0.1e-3, 0.3e-3, 0.6e-3, 1.0e-3]   # 0 = rectangular pulse
FALL_EQUALS_RISE = True              # True: symmetric trapezoid. False: always an abrupt fall
RAMP_STEP_s      = 10e-6             # a ramp is simulated as a staircase of steps this long (2 time
                                     # steps of the MRG model); halve it to check that nothing changes

# Anodal currents explored in section 7, log-spaced (each one ~1.3 x the previous)
ANODE_AMPS_A = np.geomspace(10e-6, 2e-3, 20)               # 10 uA ... 2 mA

# Section 6: amplitude of the single-fibre example runs, as a multiple of the stimulation current
SHOW_FACTOR = 8

# Section 8: settings re-checked on ALL fibres. None = automatic: the N_CHECK settings of section 7
# with the lowest charge among those that block on the example fibre. Or give them by hand, e.g.
#   D_SETTINGS = [(0.6e-3, 300e-6), (1.0e-3, 1e-3)]     # (ramp [s], anodal current [A])
D_SETTINGS = None
N_CHECK = 3

# Section 9 (textbook bipolar pair): pulse widths and amplitudes (same current in cathode and anode)
BIPOLAR_WIDTHS_s = [50e-6, 0.5e-3, 1.0e-3]
BIPOLAR_AMPS_A   = np.geomspace(5e-6, 1e-3, 12)

# Worker processes for the batch runs (1 = everything in this process)
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)

# --- consistency checks -------------------------------------------------------------------------------
assert T_PROX_ON_s + PLATEAU_START_s - max(RISE_TIMES_s) >= 0.2e-3, \
    "the longest ramp would start before t = 0.2 ms (fibre not settled): increase T_PROX_ON_s"
assert MRG["nodeAlignment"] == "proximalStation", \
    "this notebook shares ONE fibre object between all fibre lines: it needs nodeAlignment = proximalStation"
print(f"fibre D = {FIBRE_DIAM_um} um (internode {MRG_TABLE[FIBRE_DIAM_um][5]} um), {N_WORKERS} worker processes")

## 4. Data: the FEM potential along each fibre

`results/block/UA01_fibre_profiles.h5` was written by `scripts/export_block_profiles.m` from the
solved 3D model. It contains:
- **453 fibre lines**: straight lines parallel to the nerve axis, one every 100 µm inside the
  fascicles, each labelled with its fascicle;
- for **each contact** (1–6 solved, 43–48 = mirror of 1–6) and each fibre line, the potential
  $V_c(z)$ produced by **1 A** on that contact alone, sampled every 50 µm over |z| ≤ 33 mm.

The potential for any pattern and amplitude is then a weighted sum of these profiles (superposition).
The electrode layout comes from `data/electrode/sites.mat` (see its README). At the end of the cell
the anode pattern is built from the proximal one.

In [ ]:
H5_PATH = BLOCK_DIR / f"{PATIENT}_fibre_profiles.h5"
with h5py.File(H5_PATH, "r") as f:
    # MATLAB writes vectors as 1 x n matrices: ravel() turns them into flat numpy arrays
    z_m           = f["z_m"][:].ravel()                        # sample positions along the nerve [m]
    fibre_x_m     = f["x_m"][:].ravel()                        # position of each fibre line in the section [m]
    fibre_y_m     = f["y_m"][:].ravel()
    fascicle      = f["fascicle"][:].ravel().astype(int)       # fascicle containing each fibre (1...10)
    contact_id    = f["contact_id"][:].ravel().astype(int)     # contacts in the file: 1-6 and 43-48
    mirrored_from = f["mirrored_from"][:].ravel().astype(int)  # 0 = solved; k = mirror image of contact k
    contact_z_m   = f["contact_z_m"][:].ravel()                # z of each contact [m]
    n_contacts, n_fibres, n_z = f["V_perA"].shape              # the potentials themselves: loaded by fibre_jobs
    fem_source = f.attrs["source"]
fem_source = fem_source.decode() if isinstance(fem_source, bytes) else str(fem_source)
CONTACT_Z = dict(zip(contact_id.tolist(), contact_z_m.tolist()))

print(f"FEM solution: {fem_source}")
print(f"{n_fibres} fibres in {fascicle.max()} fascicles; {n_z} samples along z, "
      f"{z_m[0]*1e3:.1f} ... {z_m[-1]*1e3:.1f} mm; contacts {contact_id.tolist()}")

# Electrode layout: data/electrode/sites.mat, lengths in mm, angles in degrees (columns: README there)
sites = scipy.io.loadmat(str(ROOT / "data" / "electrode" / "sites.mat"))["sites"]
sites = pd.DataFrame(sites, columns=["AS", "type", "section", "z_mm", "r_mm", "theta_deg", "phi_deg",
                                     "thickness_mm", "sideZ_mm", "sideXY_mm"])
sites["AS"] = sites["AS"].astype(int)
sites = sites.set_index("AS")
sites_all = sites.copy()                                      # all 48 contacts of the layout
sites = sites.loc[contact_id]                                 # the contacts of the profile file
sites["kind"] = np.where(sites["type"] == 1, "shank (intraneural)", "cuff pad (nerve surface)")
CONTACT_AREA_m2 = float(sites["sideZ_mm"].iloc[0] * sites["sideXY_mm"].iloc[0]) * 1e-6   # one face
print(f"active-site face {sites['sideZ_mm'].iloc[0]:.1f} x {sites['sideXY_mm'].iloc[0]:.1f} mm = "
      f"{CONTACT_AREA_m2*1e6:.2f} mm^2 (used for the charge density)")


def layout_partners(sites_all):
    """Proximal contact -> its partner on the other site, from the nominal layout (sites.mat): same
    type, same radius and angle, opposite station (section -k -> +k). The pads of the Gmsh model are
    moved by the random lead insertion, so this is the "same electrode" of the device, not a geometric
    mirror. (With the old PDE Toolbox model and 24 solved contacts this equals the mirror partners.)"""
    out = {}
    for c, s in sites_all[sites_all.section < 0].iterrows():
        cand = sites_all[(sites_all.section == -s.section) & (sites_all.type == s.type) &
                         (np.isclose(sites_all.theta_deg, s.theta_deg)) &
                         ((sites_all.r_mm.isna() & np.isnan(s.r_mm)) | np.isclose(sites_all.r_mm, s.r_mm))]
        if len(cand) == 1:
            out[int(c)] = int(cand.index[0])
    return out

# --- The anode pattern: the SAME contacts on the distal site, polarity reversed (supervisor 2026-11-26)
MIRROR = layout_partners(sites_all)                           # proximal -> distal "same electrode"
missing = [c for c in PROX_PATTERN if c not in MIRROR or MIRROR[c] not in CONTACT_Z]
assert not missing, f"no solved distal partner for contacts {missing} in the profile file"
ANODE_PATTERN = {MIRROR[c]: -w for c, w in PROX_PATTERN.items()}
assert all(w > 0 for w in ANODE_PATTERN.values()), "the proximal weights must all be cathodic (< 0)"
PROX_NAME = "+".join(f"AS{c}" for c in PROX_PATTERN)
ANODE_NAME = "+".join(f"AS{c}" for c in ANODE_PATTERN)
print(f"proximal cathode {PROX_PATTERN}  ->  distal anode {ANODE_PATTERN}")

sites[["kind", "section", "z_mm", "r_mm", "theta_deg"]]

### How to read this figure: the cross-section

Imagine cutting the arm across the nerve and looking at the cut face. The figure is that slice.
- **Each dot is one simulated fibre.** The fibres run along the nerve, perpendicular to the screen,
  and the dot marks where a fibre crosses the slice. Every fibre is a straight line, so it is the
  same dot at every z.
- **The colour is the fascicle** (label F1…F10 at the centre of each group). The dots with a black
  outline belong to the **target fascicle** (F7).
- **The empty squares are the shank contacts.** The proximal and the distal station have the same
  contacts at the same (x, y), 20.6 mm apart along the nerve, so one square stands for both: "AS2/44"
  means AS 2 (proximal, the cathode) and AS 44 (distal, the anode).
- **The black bars at the top and bottom are the cuff pads**, drawn at an approximate position
  (`sites.mat` does not give their radius, because they sit on the nerve surface).

What to look for: the fibres closest to the active contact (AS 2/44) feel the strongest potential.
They are the first to fire when the current rises, and the ones the anode affects most.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
cmap = plt.get_cmap("tab10")
for k in range(1, fascicle.max() + 1):
    sel = fascicle == k
    ax.scatter(fibre_x_m[sel] * 1e3, fibre_y_m[sel] * 1e3, s=9, color=cmap((k - 1) % 10),
               edgecolor="k" if k == TARGET_FASCICLE else "none", linewidth=0.5)
    ax.text(fibre_x_m[sel].mean() * 1e3, fibre_y_m[sel].mean() * 1e3, f"F{k}", fontsize=9,
            weight="bold", ha="center", va="center")

y_cuff = np.abs(fibre_y_m).max() * 1e3 + 0.25                # where the cuff pads are drawn [mm]
for c, s in sites.iterrows():
    if c > 6:                                                  # draw each proximal/distal pair once
        continue
    active = c in PROX_PATTERN                                 # the contact used by this notebook
    if s["type"] == 1:                                         # shank contact: x = r cos(theta), y = r sin(theta)
        x = s["r_mm"] * np.cos(np.radians(s["theta_deg"]))
        y = s["r_mm"] * np.sin(np.radians(s["theta_deg"]))
        ax.plot(x, y, "s", ms=8, mfc="gold" if active else "none", mec="k")
        ax.annotate(f"AS{c}/{c + 42}" + (" (active)" if active else ""), (x, y), xytext=(0, 8),
                    textcoords="offset points", ha="center", fontsize=7)
    else:                                                      # cuff pad: above (theta 90) or below (270)
        y = y_cuff * np.sign(np.sin(np.radians(s["theta_deg"])))
        ax.plot([-0.1, 0.1], [y, y], "-", color="gold" if active else "k", lw=4)
        ax.annotate(f"cuff AS{c}/{c + 42} (position approx.)", (0, y), xytext=(0, 6 if y > 0 else -12),
                    textcoords="offset points", ha="center", fontsize=7)
ax.set_aspect("equal")
ax.set(xlabel="x [mm]", ylabel="y [mm]")
ax.set_title(f"{PATIENT}: fibre lines coloured by fascicle (target F{TARGET_FASCICLE} outlined); "
             "squares = shank contacts, gold = active", fontsize=9)
fig.savefig(OUT_DIR / "cross_section.png", dpi=130, bbox_inches="tight")
plt.show()

## 5. Proximal stimulation: threshold, stimulation current, example fibre

**Start the runner.** This builds the MRG fibre (60 mm, centred on z = 0, one node exactly at the
proximal station) in this process and in each worker process. APs are counted at the node 5 mm from
each end of the fibre: an AP at the proximal end is a signal to the brain (the wanted sensation), an
AP at the distal end is a signal to the muscle (what we want to block).

In [ ]:
Z_PROX_m = CONTACT_Z[next(iter(PROX_PATTERN))]
Z_DIST_m = CONTACT_Z[next(iter(ANODE_PATTERN))]
assert all(CONTACT_Z[c] < 0 for c in PROX_PATTERN), "proximal contacts must be at z < 0 (D-12)"
assert all(CONTACT_Z[c] > 0 for c in ANODE_PATTERN), "anode contacts must be at z > 0"

t0 = time.time()
runner = fibre_jobs.FibreRunner(H5_PATH, MRG, FIBRE_DIAM_um, Z_PROX_m, MECH_DIR, N_WORKERS)
FIBRE = runner.fibre
print(f"MRG fibre: {FIBRE.n_nodes} nodes, {len(FIBRE.sections)} compartments, a node every "
      f"{FIBRE.internode_m*1e3:.2f} mm, one at z = {Z_PROX_m*1e3:.1f} mm")
print(f"APs counted at z = {FIBRE.z_node_m[runner.node_brain]*1e3:.1f} mm (brain) and "
      f"z = {FIBRE.z_node_m[runner.node_muscle]*1e3:.1f} mm (muscle); started in {time.time()-t0:.1f} s")

**Pulse shapes.** `mrg_fibre` applies rectangular pulses. A trapezoid is written as a staircase of
short steps: the rising ramp is cut into steps of `RAMP_STEP_s`, and each step takes the value of
the linear ramp at its mid-point, so the charge is the same as that of the true ramp. Overlapping
pulses add up, so any shape can be built this way.

In [ ]:
def rect(t_on_s, width_s, amp_A):
    """One rectangular pulse, as the list of (t_on, t_off, amplitude) that mrg_fibre expects."""
    return [(t_on_s, t_on_s + width_s, amp_A)]


def trapezoid(t_plateau_on_s, t_plateau_off_s, amp_A, rise_s, fall_s, step_s=RAMP_STEP_s):
    """Trapezoidal pulse as a staircase of rectangular steps.

    The rise ENDS at t_plateau_on_s, the plateau (full amplitude) lasts until t_plateau_off_s, and
    the fall STARTS there. rise_s = 0 or fall_s = 0 gives an abrupt edge.
    """
    pulses = []
    n = int(round(rise_s / step_s))
    if n > 0:                                     # rising staircase
        edges = t_plateau_on_s - rise_s + np.arange(n + 1) * rise_s / n
        pulses += [(edges[k], edges[k + 1], amp_A * (k + 0.5) / n) for k in range(n)]
    pulses.append((t_plateau_on_s, t_plateau_off_s, amp_A))          # plateau
    n = int(round(fall_s / step_s))
    if n > 0:                                     # falling staircase
        edges = t_plateau_off_s + np.arange(n + 1) * fall_s / n
        pulses += [(edges[k], edges[k + 1], amp_A * (1 - (k + 0.5) / n)) for k in range(n)]
    return pulses


def charge_C(pulses):
    """Charge of one phase [C] = time integral of the current."""
    return sum((off - on) * amp for on, off, amp in pulses)


def end_s(pulses):
    return max(off for _, off, _ in pulses)


def prox_drive(amp_A):
    """The proximal stimulation channel: pattern + one rectangular cathodic pulse at T_PROX_ON_s."""
    return (PROX_PATTERN, rect(T_PROX_ON_s, PROX_WIDTH_s, amp_A))


def anode_pulses(amp_A, rise_s):
    """The anodal pulse of the distal site: plateau PLATEAU_START...PLATEAU_END after the proximal onset."""
    fall_s = rise_s if FALL_EQUALS_RISE else 0.0
    return trapezoid(T_PROX_ON_s + PLATEAU_START_s, T_PROX_ON_s + PLATEAU_END_s, amp_A, rise_s, fall_s)


def t_stop_for(*pulse_lists):
    """Simulated time: until the end of the last pulse plus T_AFTER_s."""
    return max(end_s(p) for p in pulse_lists) + T_AFTER_s


def max_density_uC_cm2(pattern, pulses):
    """Charge density per phase on the most loaded contact of the pattern [uC/cm^2]."""
    return charge_C(pulses) * max(abs(w) for w in pattern.values()) / (CONTACT_AREA_m2 * 1e4) * 1e6


T_STOP_PROX_s = T_PROX_ON_s + PROX_WIDTH_s + T_AFTER_s    # proximal pulse alone
print("example: anode 100 uA with 0.3 ms ramps ->", len(anode_pulses(100e-6, 0.3e-3)), "steps, charge",
      f"{charge_C(anode_pulses(100e-6, 0.3e-3))*1e9:.1f} nC")

**Threshold of the target fascicle and stimulation current.** For each fibre of the target fascicle,
the smallest cathodic current on the proximal pattern that fires it (50 µs pulse; doubling, then
bisection to 2 %). The fascicle threshold is the current that fires **10 % of its fibres**, the
criterion of Zelechowski 2020. The stimulation current is `stimFactor` (1.2) times that value: a
sensation just above threshold. It is the same procedure as `run_block_study.py`, so the numbers
should match the first round (8.6 µA, I_stim = 10.3 µA).

In [ ]:
t0 = time.time()
tgt = np.flatnonzero(fascicle == TARGET_FASCICLE)              # fibres of the target fascicle
jobs = [(int(i), PROX_PATTERN, T_PROX_ON_s, PROX_WIDTH_s, T_STOP_PROX_s,
         P["searchStart_A"], P["searchMax_A"], P["searchTol"]) for i in tgt]
th_prox_A = np.full(n_fibres, np.nan)                         # NaN = not computed / does not fire
th_prox_A[tgt] = runner.map(fibre_jobs.threshold_job, jobs)

n_need = max(1, math.ceil(P["recruitFraction"] * len(tgt)))    # 10 % of the fibres of the fascicle
finite = np.sort(th_prox_A[tgt][np.isfinite(th_prox_A[tgt])])
assert len(finite) >= n_need, f"F{TARGET_FASCICLE} is not excited up to {P['searchMax_A']*1e6:.0f} uA"
TH10_A = finite[n_need - 1]
I_STIM = P["stimFactor"] * TH10_A
print(f"F{TARGET_FASCICLE}: {len(tgt)} fibres, thresholds {finite.min()*1e6:.1f} ... {finite.max()*1e6:.1f} uA")
print(f"10%-of-fibres threshold {TH10_A*1e6:.2f} uA = {TH10_A*PROX_WIDTH_s*1e9:.2f} nC; "
      f"I_stim = {P['stimFactor']} x = {I_STIM*1e6:.2f} uA ({I_STIM*PROX_WIDTH_s*1e9:.2f} nC)   "
      f"[{time.time()-t0:.0f} s]")

**Which fibres does the proximal pulse excite at I_stim?** One run per fibre, all 453 fibres. These
are the reference outcomes: with the anode on, every fibre should send the **same** number of APs to
the brain as here, and **none** to the muscle.

In [ ]:
t0 = time.time()
res = runner.run([(int(i), [prox_drive(I_STIM)], T_STOP_PROX_s) for i in range(n_fibres)])
EXPECT_BRAIN = np.array([r[0] for r in res])                  # APs to the brain with the proximal pulse alone
EXPECT_MUSCLE = np.array([r[1] for r in res])
EXCITED = (EXPECT_BRAIN + EXPECT_MUSCLE) > 0
per_fasc = {f"F{k}": int(np.sum(EXCITED & (fascicle == k))) for k in range(1, fascicle.max() + 1)
            if np.any(EXCITED & (fascicle == k))}
print(f"at I_stim the proximal pulse excites {EXCITED.sum()} fibres: {per_fasc}; each sends "
      f"{set(EXPECT_BRAIN[EXCITED].tolist())} AP(s) to the brain and {set(EXPECT_MUSCLE[EXCITED].tolist())} "
      f"to the muscle   [{time.time()-t0:.0f} s]")

**Example fibre, conduction velocity and arrival time.** The example fibre is the most excitable
fibre of the target fascicle: the one closest to AS 2 and, by the mirror symmetry, to AS 44. From
its run with the proximal pulse alone we measure the conduction velocity, and the time the AP needs
to reach the distal site. The anode plateau must cover that arrival time.

### How to read this figure: the anodal pulse shapes

This is a plot of **current against time**, like an oscilloscope trace of the stimulator output.
- **x axis:** time in ms, counted from the start of the proximal pulse (0 = the stimulus that gives
  the sensation).
- **y axis:** anodal current divided by its amplitude, so 1 means full current. The real current is
  this number times the amplitude chosen in section 7.
- **Each coloured line is one ramp length.** "rise 0.0 ms" is a rectangle: the current jumps to full
  value and back with vertical edges. The longer the ramp, the more gently the current rises before
  the plateau and falls after it. The lines are drawn as staircases because that is how the ramps
  are simulated; the steps are tiny.
- **The grey bar at 0** is the proximal cathodic pulse (50 µs).
- **The red dotted line** is the moment the AP from the proximal site reaches the distal site.

What to check: **every line must be at 1 at the red line.** That means the anode is fully on when
the AP passes. If it is not, change `PLATEAU_START_s` / `PLATEAU_END_s`.

In [ ]:
cand = np.flatnonzero(EXCITED & (fascicle == TARGET_FASCICLE))
assert len(cand), "no fibre of the target fascicle is excited at I_stim"
I_EX = int(cand[np.nanargmin(th_prox_A[cand])])
nb, nm, rec = runner.record(I_EX, [prox_drive(I_STIM)], T_STOP_PROX_s)


def first_crossing_s(rec, node):
    """Time of the first upward crossing of the AP threshold (0 mV) at a node; NaN if none."""
    v = rec["v_V"][node]
    up = (v[:-1] < MRG["apThreshold_V"]) & (v[1:] >= MRG["apThreshold_V"])
    return rec["t_s"][1:][up][0] if up.any() else np.nan


# velocity between two nodes on the distal side: 5 mm before and 10 mm after the distal site
n_a, n_b = FIBRE.node_index(Z_DIST_m - 5e-3), FIBRE.node_index(Z_DIST_m + 10e-3)
CV_mps = (FIBRE.z_node_m[n_b] - FIBRE.z_node_m[n_a]) / (first_crossing_s(rec, n_b) - first_crossing_s(rec, n_a))
ARRIVAL_s = first_crossing_s(rec, FIBRE.node_index(Z_DIST_m)) - T_PROX_ON_s
print(f"example fibre {I_EX} (F{fascicle[I_EX]}, x = {fibre_x_m[I_EX]*1e3:.2f} mm, y = {fibre_y_m[I_EX]*1e3:.2f} mm), "
      f"threshold {th_prox_A[I_EX]*1e6:.2f} uA; proximal alone: {nb} AP to the brain, {nm} to the muscle")
print(f"conduction velocity {CV_mps:.1f} m/s; the AP reaches the distal site {ARRIVAL_s*1e3:.2f} ms after "
      f"the proximal onset")
if not (PLATEAU_START_s < ARRIVAL_s < PLATEAU_END_s):
    print("WARNING: the anode plateau does not contain the arrival time: change PLATEAU_START_s / PLATEAU_END_s")

# the anodal pulse shapes that will be used, normalised to their amplitude
fig, ax = plt.subplots(figsize=(8, 2.8))
for r in RISE_TIMES_s:
    p = anode_pulses(1.0, r)
    tt = np.array([[on, off] for on, off, _ in p]).ravel()
    aa = np.array([[a, a] for _, _, a in p]).ravel()
    ax.plot((tt - T_PROX_ON_s) * 1e3, aa, label=f"rise {r*1e3:.1f} ms")
ax.axvspan(0, PROX_WIDTH_s * 1e3, color="k", alpha=0.3, label="proximal cathodic pulse")
ax.axvline(ARRIVAL_s * 1e3, color="r", ls=":", label="AP arrives at the distal site")
ax.set(xlabel="time from the proximal onset [ms]", ylabel="anodal current / amplitude",
       title=f"Anodal pulse shapes on {ANODE_NAME} (trapezoids written as staircases)")
ax.legend(fontsize=7, ncol=2)
fig.savefig(OUT_DIR / "pulse_shapes.png", dpi=130, bbox_inches="tight")
plt.show()

## 6. Q1: why does the anode fire the fibre before it blocks it?

**Static view: the activating function.** For 1 µA on the cathode and 1 µA on the anode, we compute
$V_e$ at the nodes of the example fibre and its second difference $f_n$.

### How to read this figure: potential and activating function along the fibre

The two panels share the horizontal axis: **position along the nerve** in mm, with the brain to the
left and the muscle to the right. The dashed grey vertical lines mark the proximal site
(−10.3 mm, cathode) and the distal site (+10.3 mm, anode). The fibre is drawn as if stretched along
this axis.

- **Top panel, $V_e$ (the input):** the extracellular potential along the example fibre for 1 µA.
  The black curve (cathode) is a negative dip centred on the proximal site. The red curve (anode)
  is a positive bump centred on the distal site, the mirror image of the dip. Both are smooth and
  wide: the potential spreads over several mm.
- **Bottom panel, the activating function (what the membrane feels):** one dot per node of Ranvier
  (nodes are 1.15 mm apart). It measures how curved the top curve is at each node. **Above zero:**
  the stimulus pushes current into that node and depolarises it, towards firing. **Below zero:** it
  hyperpolarises it, away from firing.
  - At the proximal site the black dots form one tall positive spike: this is where the cathode
    starts the AP.
  - At the distal site the red dots form a **deep negative dot** in the middle, the hyperpolarised
    region that should block the AP, with **positive dots on both sides**: the **virtual cathodes**.
    These depolarise the fibre, so the anode can fire it there.

Notice that the anode's positive side dots are smaller than its negative centre, but they are
positive, and a node only needs to be depolarised by a certain amount to fire. If the virtual
cathodes reach that amount before the centre is negative enough to stop an incoming AP, the anode
fires the fibre before it can block it. That is the answer to Q1, and the table puts numbers on it.
The units (V/m² per µA) are not intuitive; only the ratios between the numbers matter.

The table's last column is a very rough estimate of the anodal current that fires the fibre
through the virtual cathodes: the cathodic threshold times (cathode peak / virtual-cathode peak).
It ignores that the anodal pulse lasts ~16 × longer than the 50 µs cathodic pulse, and a longer
pulse fires at a lower current (strength–duration). The real excitation current is therefore lower.

In [ ]:
def ve_at_nodes(i_fibre, pattern, amp_A=1e-6):
    """Extracellular potential [V] at the nodes of the shared fibre for amp_A of the pattern."""
    return np.interp(FIBRE.z_node_m, z_m, fibre_jobs.pattern_profile(i_fibre, pattern)) * amp_A


def activating_function(ve_nodes):
    """Second difference of V_e along the nodes divided by internode^2 [V/m^2]; NaN at the end nodes."""
    f = np.full_like(ve_nodes, np.nan)
    f[1:-1] = (ve_nodes[:-2] - 2 * ve_nodes[1:-1] + ve_nodes[2:]) / FIBRE.internode_m ** 2
    return f


zn_mm = FIBRE.z_node_m * 1e3
near_prox = np.abs(FIBRE.z_node_m - Z_PROX_m) < 3.5e-3        # +- 3 internodes around each site
near_dist = np.abs(FIBRE.z_node_m - Z_DIST_m) < 3.5e-3
af_cath = activating_function(ve_at_nodes(I_EX, PROX_PATTERN))
af_anod = activating_function(ve_at_nodes(I_EX, ANODE_PATTERN))
cath_peak = np.nanmax(af_cath[near_prox])                     # depolarisation under the cathode
vc_peak = np.nanmax(af_anod[near_dist])                       # virtual cathodes next to the anode
hyp_peak = np.nanmin(af_anod[near_dist])                      # hyperpolarisation under the anode

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for pat, name, col in [(PROX_PATTERN, f"proximal cathode {PROX_NAME}", "k"),
                       (ANODE_PATTERN, f"distal anode {ANODE_NAME}", "tab:red")]:
    a1.plot(z_m * 1e3, fibre_jobs.pattern_profile(I_EX, pat) * 1e-6 * 1e3, color=col, label=name)
    a2.plot(zn_mm, activating_function(ve_at_nodes(I_EX, pat)), ".-", color=col, label=name)
for ax in (a1, a2):
    for zz in (Z_PROX_m, Z_DIST_m):
        ax.axvline(zz * 1e3, color="grey", ls="--", lw=0.8)
    ax.axhline(0, color="grey", lw=0.5)
    ax.legend(fontsize=8)
a1.set(ylabel="V_e for 1 uA [mV]", title=f"Fibre {I_EX} (F{fascicle[I_EX]}): extracellular potential along the fibre")
a2.set(xlabel="z along the nerve [mm]   (brain <-  |  -> muscle)", ylabel="activating function [V/m^2 per uA]",
       title="Activating function at the nodes: > 0 depolarises (towards firing), < 0 hyperpolarises")
a2.set_xlim(Z_PROX_m * 1e3 - 6, Z_DIST_m * 1e3 + 6)
fig.tight_layout()
fig.savefig(OUT_DIR / "activating_function.png", dpi=130, bbox_inches="tight")
plt.show()

AF_TABLE = pd.DataFrame([{
    "cathode peak [V/m^2 per uA]": cath_peak,
    "anode: virtual-cathode peak": vc_peak,
    "anode: hyperpolarisation": hyp_peak,
    "|hyp| / virtual cathode": abs(hyp_peak) / vc_peak,
    "virtual cathode / cathode peak": vc_peak / cath_peak,
    "rough excitation estimate [uA]": th_prox_A[I_EX] * 1e6 * cath_peak / vc_peak}])
AF_TABLE.to_csv(OUT_DIR / "activating_function_table.csv", index=False)
AF_TABLE.round(3)

**Dynamic view: four runs of the example fibre.** The four cases:
1. the proximal cathode alone: one AP up (to the brain) and one down (to the muscle);
2. the anode **alone**, rectangular, at `SHOW_FACTOR` × I_stim: does it fire the fibre?
3. proximal + the same rectangular anode: is the descending AP stopped at the distal site?
4. proximal + the same anode with the longest ramps: does the ramp change the picture?

### How to read this figure: the raster (space–time map of the fibre)

This is the most important type of plot in the block study. It shows **the whole fibre over the
whole simulation** in one image.
- **x axis = time** (ms from the proximal onset).
- **y axis = position along the fibre** (mm). Each horizontal row is one node of Ranvier. The brain
  end is at the bottom, the muscle end at the top. The two dashed cyan lines are the proximal site
  (cathode, −10.3 mm) and the distal site (anode, +10.3 mm).
- **Colour = membrane voltage of that node at that time** (scale bar on the right):
  dark purple ≈ −80 mV = **at rest**; bright yellow/white ≈ +20…+40 mV = **firing (an AP)**;
  black (below −90 mV) = **hyperpolarised**.

**How to recognise what happens:**
- **An AP is a bright stripe.** Because it moves, the stripe is **tilted**: a stripe going up and to
  the right is an AP travelling towards the muscle, and one going down and to the right is travelling
  towards the brain. The slope is the conduction velocity: a 50 m/s AP covers 50 mm in 1 ms.
- **An AP born at a site** looks like a **">" shape** with its tip on that site's line: two stripes
  leaving it, one up and one down. This is panel 1 (born at the proximal site).
- **A blocked AP** is a stripe that **ends** at the distal line and does not continue above it.
- **A new AP fired by the anode** is a ">" whose tip is at the distal site (or just above or below
  it, at the virtual cathodes), usually at the moment the anode switches on (or off, for anode
  break).
- **The anode itself is visible**: while it is on, the rows at the distal site turn **black**
  (hyperpolarised), and the rows just above and below it turn **lighter** (depolarised, the virtual
  cathodes).
- **Counting:** a stripe that reaches the **top edge** is an AP delivered to the muscle; one that
  reaches the **bottom edge** is an AP delivered to the brain. The panel title gives the two counts.

A real example from the first round is in `results/block/UA01_AS2_D10_raster.png` (local file, not
in git). Panel 1 is the ">" from the proximal site. In panel 2 (collision) the descending AP and an
AP coming up from the distal site meet and vanish between the sites, but a new stripe still goes up
from the distal site to the muscle. In panel 3 (cathodal block) the upward stripe stops at the
distal line. In panel 4 (anode on AS 44) there is a black band at the distal site, bright regions
just above and below it at the anode onset (virtual cathodes firing), and a stripe that reaches the
muscle.

**The goal, in raster terms:** a stripe that goes down to the bottom edge (sensation) and a stripe
going up that **stops at the distal line**, with **no new ">" at the distal site**.

In [ ]:
def plot_raster(ax, rec, title):
    """Membrane voltage at the nodes vs time, with the two sites marked."""
    t, v = rec["t_s"], rec["v_V"]
    im = ax.pcolormesh((t - T_PROX_ON_s) * 1e3, FIBRE.z_node_m * 1e3, v * 1e3, cmap="magma",
                       vmin=-100, vmax=40, shading="auto")
    for zz, lab in [(Z_PROX_m, "proximal (cathode)"), (Z_DIST_m, "distal (anode)")]:
        ax.axhline(zz * 1e3, color="c", lw=0.8, ls="--")
        ax.text((t[-1] - T_PROX_ON_s) * 1e3, zz * 1e3, lab, color="c", ha="right", va="bottom", fontsize=7)
    ax.set_title(title, fontsize=8.5)
    return im


A_SHOW = SHOW_FACTOR * I_STIM
r_show = max(RISE_TIMES_s)
t_stop = t_stop_for(rect(T_PROX_ON_s, PROX_WIDTH_s, I_STIM), anode_pulses(A_SHOW, r_show))
cases = [("proximal cathode alone", [prox_drive(I_STIM)]),
         (f"anode {ANODE_NAME} ALONE, rectangular", [(ANODE_PATTERN, anode_pulses(A_SHOW, 0.0))]),
         (f"proximal + anode {ANODE_NAME}, rectangular",
          [prox_drive(I_STIM), (ANODE_PATTERN, anode_pulses(A_SHOW, 0.0))]),
         (f"proximal + anode {ANODE_NAME}, {r_show*1e3:.1f} ms ramps",
          [prox_drive(I_STIM), (ANODE_PATTERN, anode_pulses(A_SHOW, r_show))])]

fig, axs = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
for ax, (title, drives) in zip(axs.ravel(), cases):
    nb, nm, rec = runner.record(I_EX, drives, t_stop)
    im = plot_raster(ax, rec, f"{title}\n-> {nb} AP to the brain, {nm} to the muscle")
for ax in axs[1]:
    ax.set_xlabel("time from the proximal onset [ms]")
for ax in axs[:, 0]:
    ax.set_ylabel("node position z [mm]\n(brain bottom, muscle top)")
fig.colorbar(im, ax=axs, label="membrane potential at the nodes [mV]")
fig.suptitle(f"Fibre {I_EX} (F{fascicle[I_EX]}), MRG {FIBRE_DIAM_um} um; anode {A_SHOW*1e6:.0f} uA "
             f"(= {SHOW_FACTOR} x I_stim), plateau {PLATEAU_START_s*1e3:.1f}-{PLATEAU_END_s*1e3:.1f} ms", fontsize=10)
fig.savefig(OUT_DIR / "raster_four_cases.png", dpi=130, bbox_inches="tight")
plt.show()

## 7. Q2: map of ramp × anodal current (example fibre)

For every ramp time and anodal current we run the example fibre **twice**:
- **anode alone**: does the anode fire the fibre by itself (virtual cathodes or anode break)?
- **proximal + anode**: is the descending AP stopped?

Each setting gets one of four outcomes:

| outcome | meaning | colour | letter |
|---|---|---|---|
| **anode fires** | the anode alone sends at least one AP somewhere: it is a stimulus, not a block | red | F |
| **blocked** | anode alone silent, and with the proximal pulse: no AP to the muscle and the same APs to the brain as the proximal pulse alone. **This is what we want.** | green | B |
| **passes** | anode alone silent, but the descending AP still reaches the muscle: the anode is too weak | grey | P |
| **other** | anything else (e.g. the AP to the brain is lost or doubled) | yellow | O |

Number of runs = ramps × currents × 2.

In [ ]:
CATEGORIES = ["passes", "blocked", "anode fires", "other"]
CAT_COLOURS = ["#9e9e9e", "#2e7d32", "#c62828", "#f9a825"]
CAT_LETTERS = ["P", "B", "F", "O"]


def categorise(alone, combined, expect_brain):
    """Outcome of one setting on one fibre, from the (n_brain, n_muscle) of the two runs."""
    if sum(alone) > 0:
        return "anode fires"
    if combined[1] == 0 and combined[0] == expect_brain:
        return "blocked"
    if combined[1] > 0:
        return "passes"
    return "other"


def setting_jobs(i_fibre, pulses):
    """The two jobs of one setting: anode alone, and proximal + anode (same simulated time)."""
    t_stop = t_stop_for(rect(T_PROX_ON_s, PROX_WIDTH_s, I_STIM), pulses)
    return [(int(i_fibre), [(ANODE_PATTERN, pulses)], t_stop),
            (int(i_fibre), [prox_drive(I_STIM), (ANODE_PATTERN, pulses)], t_stop)]


grid = [(rise, amp) for rise in RISE_TIMES_s for amp in ANODE_AMPS_A]
jobs = []
for rise, amp in grid:
    jobs += setting_jobs(I_EX, anode_pulses(amp, rise))
print(f"{len(grid)} settings x 2 runs = {len(jobs)} simulations on {N_WORKERS} processes...")
t0 = time.time()
out = runner.run(jobs)
print(f"done in {time.time()-t0:.0f} s")

rows = []
for k, (rise, amp) in enumerate(grid):
    alone, comb = out[2 * k], out[2 * k + 1]
    pulses = anode_pulses(amp, rise)
    rows.append({"rise_s": rise, "amp_A": amp, "rise_ms": rise * 1e3, "amp_uA": amp * 1e6,
                 "x_Istim": amp / I_STIM,
                 "alone_brain": alone[0], "alone_muscle": alone[1],
                 "comb_brain": comb[0], "comb_muscle": comb[1],
                 "outcome": categorise(alone, comb, EXPECT_BRAIN[I_EX]),
                 "charge_nC": charge_C(pulses) * 1e9,
                 "max_density_uC_cm2": max_density_uC_cm2(ANODE_PATTERN, pulses)})
MAP = pd.DataFrame(rows)
MAP.to_csv(OUT_DIR / "map_example_fibre.csv", index=False)
MAP["outcome"].value_counts()

### How to read this figure: the outcome map

This is a **chessboard of experiments**. Each cell is one setting, i.e. one pair of simulations of
the example fibre.
- **Column = anodal current**, weak on the left and strong on the right. The scale is logarithmic:
  each column is about 1.3 × the previous one, so the right edge is 200 × the left edge.
- **Row = ramp length**: rectangular pulse at the bottom, the longest ramp at the top.
- **Colour (and letter) = outcome**, from the table above.

**What to expect, reading one row from left to right:** grey (P) at low current, because the anode
is too weak and the AP passes; then, as the current grows, either green (B, block) or red (F, the
anode fires the fibre). A **window** is a run of green cells between the grey and the red: currents
strong enough to block, not yet strong enough to fire.

**Comparing rows answers Q2:** if the bottom row (rectangular) goes straight from grey to red, as in
the first round, while the upper rows (ramped) show green cells, then the ramp is what makes the
block possible. If every row goes from grey to red, ramping is not enough on this fibre.

In [ ]:
M = np.array([[CATEGORIES.index(MAP[(MAP.rise_s == r) & (MAP.amp_A == a)].outcome.iloc[0])
               for a in ANODE_AMPS_A] for r in RISE_TIMES_s])
fig, ax = plt.subplots(figsize=(12, 1.2 + 0.55 * len(RISE_TIMES_s)))
ax.imshow(M, cmap=ListedColormap(CAT_COLOURS), vmin=-0.5, vmax=len(CATEGORIES) - 0.5, aspect="auto",
          origin="lower")
for i in range(M.shape[0]):                       # the letter of the outcome in every cell
    for j in range(M.shape[1]):
        ax.text(j, i, CAT_LETTERS[M[i, j]], ha="center", va="center", fontsize=7, color="w")
ax.set_yticks(range(len(RISE_TIMES_s)), [f"{r*1e3:.1f}" for r in RISE_TIMES_s])
ax.set_ylabel("ramp [ms]\n(0 = rectangular)")
ax.set_xticks(range(len(ANODE_AMPS_A)), [f"{a*1e6:.0f}" for a in ANODE_AMPS_A], rotation=45)
ax.set_xlabel(f"anodal current on {ANODE_NAME} [uA]   (I_stim = {I_STIM*1e6:.1f} uA)")
ax.legend(handles=[Patch(color=c, label=f"{l} ({s})") for c, l, s in zip(CAT_COLOURS, CATEGORIES, CAT_LETTERS)],
          loc="upper center", bbox_to_anchor=(0.5, 1.35), ncol=4, fontsize=8)
ax.set_title(f"Example fibre {I_EX} (F{fascicle[I_EX]}): outcome of the distal anode", fontsize=10, pad=30)
fig.savefig(OUT_DIR / "map_example_fibre.png", dpi=130, bbox_inches="tight")
plt.show()

**Windows in numbers.** One row per ramp length:
- `first_fire_uA`: the lowest current at which the anode fires the fibre by itself;
- `first_block_uA` … `last_block_uA`: the range of currents that block, and `n_blocking_amps` how
  many of the tested currents block;
- a useful ramp has `first_block_uA` **below** `first_fire_uA`;
- `charge_…` and `density_…` refer to the lowest blocking current. The density is per phase on the
  contact. Compare it with the charge-injection capacity of the contact material (gold,
  `data/electrode/README.md`; course slides 08); that comparison is **not** made here.

In [ ]:
def window_rows(df):
    rows = []
    for rise, g in df.groupby("rise_s", sort=False):
        blk, fire = g[g.outcome == "blocked"], g[g.outcome == "anode fires"]
        first = blk.iloc[0] if len(blk) else None                 # rows are in increasing current
        rows.append({"rise_ms": rise * 1e3,
                     "first_fire_uA": fire.amp_uA.min() if len(fire) else np.nan,
                     "first_block_uA": blk.amp_uA.min() if len(blk) else np.nan,
                     "last_block_uA": blk.amp_uA.max() if len(blk) else np.nan,
                     "n_blocking_amps": len(blk),
                     "charge_at_first_block_nC": first.charge_nC if first is not None else np.nan,
                     "density_at_first_block_uC_cm2": first.max_density_uC_cm2 if first is not None else np.nan})
    return pd.DataFrame(rows)


WIN = window_rows(MAP)
WIN.to_csv(OUT_DIR / "windows_example_fibre.csv", index=False)
WIN.round(1)

## 8. Q3: the best settings on all fibres

The example fibre is the **best case**: it is the fibre closest to the anode, so it gets the
strongest hyperpolarisation, and also the strongest virtual cathodes. A setting that blocks it must
now be checked on the whole nerve:
- on the fibres the proximal pulse excites: is the descending AP blocked?
- on **every** fibre, including all the other fascicles: does the anode alone fire it? A fibre of
  another fascicle fired by the anode is a new, unwanted AP (to the muscle and to the brain).

For the fibres the proximal pulse does not excite, only the "anode alone" run is needed: their
correct outcome is silence.

Settings: `D_SETTINGS` from the parameter cell, or, if it is None, the `N_CHECK` ramps with the
lowest charge among those that block the example fibre. For each ramp, the current is the blocking
current in the middle (geometric) of its blocking range, which leaves some margin on both sides.
Cost: (453 + number of excited fibres) runs per setting.

In [ ]:
settings = D_SETTINGS
if settings is None:
    cands = []
    for rise, g in MAP.groupby("rise_s", sort=False):
        blk = g[g.outcome == "blocked"]
        if len(blk) == 0:
            continue
        centre = np.exp(np.log(blk.amp_A).mean())             # geometric centre of the blocking currents
        amp = float(blk.amp_A.iloc[np.argmin(np.abs(np.log(blk.amp_A) - np.log(centre)))])  # a tested one
        cands.append((charge_C(anode_pulses(amp, rise)), float(rise), amp))
    settings = [(rise, amp) for _, rise, amp in sorted(cands)[:N_CHECK]]
print("settings to check:" if settings else "no setting blocks the example fibre: nothing to check on all fibres")
for rise, amp in settings:
    print(f"   anode {ANODE_NAME}, ramp {rise*1e3:.1f} ms, {amp*1e6:.0f} uA")

exc = np.flatnonzero(EXCITED)
POP_ROWS, FIBRE_ROWS = [], []
for rise, amp in settings:
    t0 = time.time()
    pulses = anode_pulses(amp, rise)
    t_stop = t_stop_for(rect(T_PROX_ON_s, PROX_WIDTH_s, I_STIM), pulses)
    jobs = [(int(i), [(ANODE_PATTERN, pulses)], t_stop) for i in range(n_fibres)] \
         + [(int(i), [prox_drive(I_STIM), (ANODE_PATTERN, pulses)], t_stop) for i in exc]
    out = runner.run(jobs)
    alone, comb = out[:n_fibres], dict(zip(exc.tolist(), out[n_fibres:]))
    fires = np.array([sum(a) > 0 for a in alone])                 # the anode alone fires this fibre
    status = np.where(fires, "anode fires", "quiet")
    for i in exc:
        if not fires[i]:
            status[i] = categorise(alone[i], comb[i], EXPECT_BRAIN[i])
    other = fascicle != TARGET_FASCICLE
    POP_ROWS.append({"rise_ms": rise * 1e3, "amp_uA": amp * 1e6, "x_Istim": amp / I_STIM,
                     "charge_nC": charge_C(pulses) * 1e9,
                     "max_density_uC_cm2": max_density_uC_cm2(ANODE_PATTERN, pulses),
                     "excited_by_proximal": len(exc), "blocked": int(np.sum(status[exc] == "blocked")),
                     "passes": int(np.sum(status[exc] == "passes")),
                     "anode_fires_target_fascicle": int(np.sum(fires & ~other)),
                     "anode_fires_other_fascicles": int(np.sum(fires & other)),
                     "fascicles_fired_by_anode": sorted(set(fascicle[fires].tolist()))})
    FIBRE_ROWS += [{"rise_ms": rise * 1e3, "amp_uA": amp * 1e6, "fibre": i, "fascicle": fascicle[i],
                    "excited_by_proximal": bool(EXCITED[i]), "status": status[i]} for i in range(n_fibres)]
    print(f"   ramp {rise*1e3:.1f} ms, {amp*1e6:.0f} uA: blocked {POP_ROWS[-1]['blocked']}/{len(exc)}, "
          f"anode fires {fires.sum()} fibres  [{time.time()-t0:.0f} s]")

POP = pd.DataFrame(POP_ROWS)
pd.DataFrame(FIBRE_ROWS).to_csv(OUT_DIR / "population_per_fibre.csv", index=False)
POP.to_csv(OUT_DIR / "population_summary.csv", index=False)
POP

### How to read this figure: the outcome on the cross-section

The same slice of the nerve as in section 4 (one dot = one fibre), drawn once per checked setting.
Now the **colour is what happened to that fibre** with that setting:
- **green**: the proximal pulse excited it and the anode **blocked** its AP to the muscle (wanted);
- **blue**: the proximal pulse excited it and the AP **still reached the muscle** (block failed);
- **red**: the **anode itself fired** this fibre (unwanted: a new AP to the muscle, and an extra,
  wrong sensation);
- **yellow**: other anomalies;
- **light grey**: quiet, nothing happened, which is correct for a fibre the proximal pulse does
  not excite.

What to look for: a good setting has only green and grey. Red dots show **where** the anode
spills over: usually around the active contact first, then spreading to the neighbouring fascicles
as the current grows.

In [ ]:
if len(POP_ROWS):
    FR = pd.DataFrame(FIBRE_ROWS)
    colours = {"blocked": "#2e7d32", "passes": "#5c6bc0", "other": "#f9a825", "anode fires": "#c62828",
               "quiet": "#d0d0d0"}
    fig, axs = plt.subplots(1, len(settings), figsize=(5.2 * len(settings), 4.6), squeeze=False)
    for ax, (rise, amp) in zip(axs[0], settings):
        g = FR[(FR.rise_ms == rise * 1e3) & (FR.amp_uA == amp * 1e6)].set_index("fibre")
        ax.scatter(fibre_x_m * 1e3, fibre_y_m * 1e3, s=10, c=[colours[g.status[i]] for i in range(n_fibres)])
        ax.set_aspect("equal")
        ax.set_title(f"anode {ANODE_NAME}, ramp {rise*1e3:.1f} ms, {amp*1e6:.0f} uA", fontsize=9)
        ax.set_xlabel("x [mm]")
    axs[0, 0].set_ylabel("y [mm]")
    fig.legend(handles=[Patch(color=c, label=l) for l, c in colours.items()], loc="lower center", ncol=5, fontsize=8)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    fig.savefig(OUT_DIR / "population_cross_section.png", dpi=130, bbox_inches="tight")
    plt.show()

## 9. Q4: the textbook bipolar pair (one circuit, cathode proximal, anode distal)

In a classic unidirectional cuff, the cathode and the anode belong to **one** circuit: the same
current leaves the anode and returns through the cathode, with no far ground. Here that means the
pattern `{AS 2: −1, AS 44: +1}` driven by a single pulse. These are the same contacts as in the
rest of the notebook, so the supervisor's rule holds. The AP starts under the cathode, and the anode,
on the side of the muscle, should block its descending branch.

With the poles 20.6 mm apart this should **not** work. The descending AP reaches the anode
≈ 0.4 ms after the start of the pulse, so with a 50 µs pulse the anode is already off. With a long
pulse it is still on, but the anodal current is only equal to the cathodic one, and that is
probably too weak to block (section 7 shows how much block needs). The run checks it on all the
fibres of the target fascicle, for a few pulse widths.

### How to read this figure: recruitment curves of the bipolar pair

One panel per pulse width. **x axis:** the current of the pair (log scale, weak to strong).
**y axis:** how many fibres of the target fascicle (out of the number in the label) end in each
state. As the current grows, more fibres are recruited, so the curves rise. The three lines:
- **unidirectional** (circles): APs to the brain and none to the muscle. **This is the wanted
  outcome.**
- **both ways** (squares): APs to both ends, i.e. an ordinary stimulus, not blocked;
- **muscle only** (triangles): an AP to the muscle and none to the brain, the worst case.

The pair works if the "unidirectional" line rises while "both ways" stays at zero. If "both ways"
rises first, the pair behaves like a normal stimulus.

In [ ]:
BIPOLAR = {**PROX_PATTERN, **ANODE_PATTERN}
print("bipolar pattern:", BIPOLAR, "(weights sum to", round(sum(BIPOLAR.values()), 12), ": no current to the far ground)")

bip_grid = [(w, a) for w in BIPOLAR_WIDTHS_s for a in BIPOLAR_AMPS_A]
jobs = [(int(i), [(BIPOLAR, rect(T_PROX_ON_s, w, a))], T_PROX_ON_s + w + T_AFTER_s)
        for w, a in bip_grid for i in tgt]
t0 = time.time()
out = runner.run(jobs)
print(f"{len(jobs)} simulations in {time.time()-t0:.0f} s")

rows, k = [], 0
for w, a in bip_grid:
    res = out[k:k + len(tgt)]
    k += len(tgt)
    rows.append({"width_ms": w * 1e3, "amp_uA": a * 1e6,
                 "unidirectional": sum(nb > 0 and nm == 0 for nb, nm in res),
                 "both ways": sum(nb > 0 and nm > 0 for nb, nm in res),
                 "muscle only": sum(nb == 0 and nm > 0 for nb, nm in res),
                 "none": sum(nb == 0 and nm == 0 for nb, nm in res),
                 "charge_nC": a * w * 1e9})
BIP = pd.DataFrame(rows)
BIP.to_csv(OUT_DIR / "bipolar_target_fascicle.csv", index=False)

fig, axs = plt.subplots(1, len(BIPOLAR_WIDTHS_s), figsize=(4.4 * len(BIPOLAR_WIDTHS_s), 3.6), sharey=True)
for ax, w in zip(np.atleast_1d(axs), BIPOLAR_WIDTHS_s):
    g = BIP[BIP.width_ms == w * 1e3]
    for col, st in [("unidirectional", "o-"), ("both ways", "s-"), ("muscle only", "^-")]:
        ax.semilogx(g.amp_uA, g[col], st, label=col)
    ax.set(title=f"bipolar pair {PROX_NAME}(-) / {ANODE_NAME}(+), {w*1e3:.2f} ms", xlabel="current [uA]")
np.atleast_1d(axs)[0].set_ylabel(f"fibres of F{TARGET_FASCICLE} (of {len(tgt)})")
np.atleast_1d(axs)[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "bipolar_target_fascicle.png", dpi=130, bbox_inches="tight")
plt.show()
BIP[BIP[["unidirectional", "both ways", "muscle only"]].sum(axis=1) > 0]

## 10. Summary, outputs, limitations

The cell below writes `summary.json` with the main numbers and stops the worker processes. All the
tables and figures are in `results/block/anode_cathode/`:

| file | content |
|---|---|
| `cross_section.png` | fibre lines, fascicles, contacts |
| `pulse_shapes.png` | the anodal pulse shapes and the AP arrival time |
| `activating_function.png`, `activating_function_table.csv` | section 6, static view |
| `raster_four_cases.png` | section 6, the four single-fibre runs |
| `map_example_fibre.png/.csv`, `windows_example_fibre.csv` | section 7 |
| `population_summary.csv`, `population_per_fibre.csv`, `population_cross_section.png` | section 8 |
| `bipolar_target_fascicle.png/.csv` | section 9 |
| `summary.json` | the main numbers |

**Limitations to keep in mind when reading the results**
- One fibre diameter (10 µm). Block and excitation thresholds scale differently with the diameter,
  and the arrival time depends on the conduction velocity.
- Straight, parallel fibres. A node sits exactly at the proximal station, so at the distal station
  the node offset is fixed by the internode length (best case for the proximal excitation only).
- The distal contacts are the mirror image of station −4: this is valid only while the nerve section
  is the same along z (D-12).
- Multi-contact patterns (if `PROX_PATTERN` has several contacts) are built by superposition of
  single-contact solutions in which the idle contacts are insulating (D-07). This is approximate for
  real metal contacts.
- Ramps are staircases with 10 µs steps (check by halving `RAMP_STEP_s`).
- Quasi-static tissue: acceptable for the long anodal pulses, less so for the 50 µs proximal pulse
  (perineurium corner frequency ≈ 2.6 kHz, Horn 2023).
- **Safety is not assessed.** The charge densities are reported but not compared with the Shannon
  limit or the charge-injection capacity of gold; anodal block at hundreds of µA for ~1 ms on a
  0.04 mm² contact is very likely beyond them (`docs/block.md` §6).

In [ ]:
summary = {
    "patient": PATIENT, "fem_source": fem_source, "fibre_diam_um": FIBRE_DIAM_um,
    "prox_pattern": PROX_PATTERN, "anode_pattern": ANODE_PATTERN, "target_fascicle": TARGET_FASCICLE,
    "th10_target_uA": TH10_A * 1e6, "I_stim_uA": I_STIM * 1e6,
    "excited_by_proximal": int(EXCITED.sum()), "example_fibre": I_EX,
    "conduction_velocity_mps": CV_mps, "arrival_at_distal_ms": ARRIVAL_s * 1e3,
    "anode_window_ms": [PLATEAU_START_s * 1e3, PLATEAU_END_s * 1e3],
    "rise_times_ms": [r * 1e3 for r in RISE_TIMES_s],
    "windows_example_fibre": WIN.to_dict(orient="records"),
    "population": POP.to_dict(orient="records") if len(POP_ROWS) else [],
    "bipolar_target_fascicle": BIP.to_dict(orient="records"),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
runner.close()
print("wrote", OUT_DIR / "summary.json", "and stopped the worker processes")